# Milestone 2: Preprocessing Notebook
**Renewable Energy-Aware Load Prediction Model for Smart Grids**  
*Pavagada Solar Park (2,050 MW), Tumkur District, Karnataka*

This notebook executes and validates the preprocessing pipeline implemented in `src/preprocess.py`:
1. Timestamp alignment and continuous hourly reindexing.
2. Physical constraint enforcement (solar power clipping to [0, 2050] MW, zero night solar, humidity clipping).
3. Robust rolling z-score demand outlier detection (median and MAD).
4. Gap imputation (short gaps <= 3h via linear interpolation; long gaps via seasonal lag recurrence).
5. Preservation of outage and outlier flags (`is_curtailed_or_outage`, `is_outlier`, `is_demand_outlier`).
6. Removal of unneeded constants, text metadata, and leaky target derivatives.
7. Generation of the Checkpoint Before/After missing-value comparison table.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Set up project root
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import config
from src.preprocess import run_preprocessing_pipeline, load_raw_data

# Inspect raw dataset missingness
df_raw = load_raw_data(config.RAW_DATA_FILE)
raw_nulls = df_raw.isnull().sum()
raw_null_cols = raw_nulls[raw_nulls > 0]
print(f'Raw dataset shape: {df_raw.shape}')
print(f'Columns with missing values: {len(raw_null_cols)}')

Raw dataset shape: (43824, 69)
Columns with missing values: 21


### Initial Raw Data Missingness Summary
Raw records exhibit small, localized missingness in weather (~0.36% - 0.40%) and demand series (~0.35%). Solar power generation is completely intact (0 missing values).

In [2]:
# Execute preprocessing pipeline
df_raw, df_clean = run_preprocessing_pipeline()
print(f'Preprocessed clean dataset shape: {df_clean.shape}')
print(f'Total null entries in clean dataset: {df_clean.isnull().sum().sum()}')
print(f'Total rows flagged with imputed values (is_filled=1): {df_clean["is_filled"].sum()}')
print(f'Total demand outliers flagged (is_demand_outlier=1): {df_clean["is_demand_outlier"].sum()}')

Preprocessed clean dataset shape: (43824, 47)
Total null entries in clean dataset: 0
Total rows flagged with imputed values (is_filled=1): 482
Total demand outliers flagged (is_demand_outlier=1): 80


### Preprocessing Pipeline Summary
- **Reindexed**: Guaranteed 43,824 continuous hourly observations with 0 gaps.
- **Physical Capping**: Solar strictly clamped to [0, 2050] MW and 0 at night; humidity clamped to [0, 100]%.
- **Imputation**: 482 rows had at least one missing channel filled, recorded with `is_filled`.
- **Outliers**: 80 anomalous demand hours identified with robust rolling z-scores.

In [3]:
# Milestone 2 Checkpoint: Before / After Missing-Value Table
clean_nulls = df_clean.isnull().sum()
rows = []
for col in raw_null_cols.index:
    if col in df_clean.columns:
        rows.append({
            'Column': col,
            'Raw Missing': int(raw_nulls[col]),
            'Raw Missing %': f'{raw_nulls[col] / len(df_raw) * 100:.3f}%',
            'Clean Missing': int(clean_nulls[col]),
            'Clean Missing %': f'{clean_nulls[col] / len(df_clean) * 100:.3f}%'
        })
comparison_df = pd.DataFrame(rows)
display(comparison_df)

,Column,Raw Missing,Raw Missing %,Clean Missing,Clean Missing %
0,ghi_wm2,158,0.361%,0,0.000%
1,dni_wm2,158,0.361%,0,0.000%
2,dhi_wm2,158,0.361%,0,0.000%
3,poa_irradiance_wm2,158,0.361%,0,0.000%
4,temperature_2m_c,177,0.404%,0,0.000%
5,module_temp_c,158,0.361%,0,0.000%
6,relative_humidity_pct,177,0.404%,0,0.000%
7,wind_speed_10m_ms,177,0.404%,0,0.000%
8,wind_direction_deg,177,0.404%,0,0.000%
9,cloud_cover_total_pct,177,0.404%,0,0.000%


### Missing-Value Validation
Every missing entry in the feature set has been resolved (0.000% clean missing), preserving chronological sequence length.

In [4]:
# Target identity check: Net Load == Demand - Solar
diff = np.abs((df_clean[config.TARGET_DEMAND] - df_clean[config.TARGET_SOLAR]) - df_clean[config.TARGET_NET_LOAD])
print(f'Max absolute difference between Net Load and (Demand - Solar): {diff.max():.6f} MW')
assert (diff < 1e-4).all(), 'Target identity mismatch!'
print('Target consistency check: PASSED')

Max absolute difference between Net Load and (Demand - Solar): 0.000000 MW
Target consistency check: PASSED


## Milestone 2 Checkpoint Conclusion
Milestone 2 is complete. Clean data is saved to:
- `data/interim/clean_hourly.csv`
- `data/interim/clean_hourly.parquet`

All test assertions in `tests/test_preprocess.py` pass cleanly.